In [1]:
import glob
import utils
import numpy as np
from pathlib import Path

In [2]:
current_path = Path.cwd()
parent_path = current_path.parent
data_path = parent_path / "data" / "raw" / "traffic_0.15_accident_0_steps_1000"
meta_data_paths = glob.glob(str(data_path / "**" / "*.json"), recursive=True)
data_paths = {path.split("/")[-2]: path for path in meta_data_paths}
episodes_metas = {key:utils.read_json(value) for key, value in data_paths.items()}

In [8]:
# end analysis
overtake = {"overtake_vehicle_num": []}
navigation = {"navigation_forward": [], "navigation_left": [], "navigation_right": []}
crash = {"crash_vehicle": [], "crash_object": [], "crash_building": [], "crash_human": [], "crash_sidewalk": []}
arrived = {"out_of_road": [], "arrive_dest": []}

for episode, metadata in episodes_metas.items():

    # Collect episode IDs for each category
    overtake["overtake_vehicle_num"].append(
        (episode, metadata["overtake_vehicle_num"])
    )

    navigation["navigation_"+metadata["navigation_command"]].append(episode)

    for key in crash:
        if metadata[key]:
            crash[key].append(episode)

    for key in arrived:
        if metadata[key]:
            arrived[key].append(episode)


In [9]:
z = [i[1] for i in overtake["overtake_vehicle_num"]]
print(f"Number of overtakes: {sum(z)}")

Number of overtakes: 0


In [10]:
for key, value in navigation.items():
    print(f"{key}:{len(value)}")

navigation_forward:687
navigation_left:210
navigation_right:104


In [11]:
for key, value in crash.items():
    print(f"{key}:{len(value)}")

crash_vehicle:376
crash_object:0
crash_building:0
crash_human:0
crash_sidewalk:5


In [12]:
for key, value in arrived.items():
    print(f"{key}:{len(value)}")

out_of_road:236
arrive_dest:387


In [16]:
# frame distribution
arrived_list_steps = {key:episodes_metas[key]["num_steps"] for key in arrived["arrive_dest"]}
arived_values = list(arrived_list_steps.values())
out_of_road_list_steps = {key:episodes_metas[key]["num_steps"] for key in arrived["out_of_road"]}
out_of_road_values = list(out_of_road_list_steps.values())

print("Arrived")
print(f"Mean: {np.mean(arived_values):.2f}, Std: {np.std(arived_values):.2f}")
print(f"Min: {np.min(arived_values)}, Max: {np.max(arived_values)}")
print(f"Median: {np.median(arived_values):.2f}")

print("-"*20)

print("Out of road")
print(f"Mean: {np.mean(out_of_road_values):.2f}, Std: {np.std(out_of_road_values):.2f}")
print(f"Min: {np.min(out_of_road_values)}, Max: {np.max(out_of_road_values)}")
print(f"Median: {np.median(out_of_road_values):.2f}")

print("-"*20)
total_frames = np.sum([episode["num_steps"] for episode in episodes_metas.values()])
print(f"Total number of frames: {total_frames}")

Arrived
Mean: 477.02, Std: 129.54
Min: 253, Max: 883
Median: 458.00
--------------------
Out of road
Mean: 281.22, Std: 161.30
Min: 85, Max: 833
Median: 259.00
--------------------
Total number of frames: 341781


In [17]:
# number of episodes that were never completed (i.e stuck in traffic)
incomplete_episodes = [episode for episode, meta in episodes_metas.items() if meta["num_steps"] >= 1000]
incomplete_episodes

['episode_058', 'episode_905']

## Action dataset

In [3]:
current_path = Path.cwd()
parent_path = current_path.parent
data_path = parent_path / "data" / "raw" / "traffic_0.15_accident_0_steps_1000"
action_data_paths = glob.glob(str(data_path / "**" / "actions.npy"), recursive=True)
action_paths = {path.split("/")[-2]: path for path in action_data_paths}
episodes_actions = {key:np.load(value) for key, value in action_paths.items()}

In [6]:
episode_id = "episode_094"
print(f"Number of steps (metadata): {episodes_metas[episode_id]['num_steps']}")
print(f"Actions shape: {episodes_actions[episode_id].shape}")
print(f"Number of actions: {len(episodes_actions[episode_id])}")

Number of steps (metadata): 178
Actions shape: (178, 2)
Number of actions: 178


## State dataset

In [18]:
current_path = Path.cwd()
parent_path = current_path.parent
data_path = parent_path / "data" / "raw" / "traffic_0.15_accident_0_steps_1000"
state_data_paths = glob.glob(str(data_path / "**" / "states.npy"), recursive=True)
state_paths = {path.split("/")[-2]: path for path in state_data_paths}
episodes_states = {key:np.load(value) for key, value in state_paths.items()}

In [20]:
episode_id = "episode_094"
print(f"Number of steps (metadata): {episodes_metas[episode_id]['num_steps']}")
print(f"Actions shape: {episodes_actions[episode_id].shape}")
print(f"Number of actions: {len(episodes_actions[episode_id])}, shape {episodes_actions[episode_id].shape}")
print(f"Number of states: {len(episodes_states[episode_id])}, shape {episodes_states[episode_id].shape}")

Number of steps (metadata): 178
Actions shape: (178, 2)
Number of actions: 178, shape (178, 2)
Number of states: 178, shape (178, 19)
